In [328]:
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.model_selection import train_test_split
import catboost
from scipy import stats 
import numpy as np
from sklearn.metrics import root_mean_squared_error
import warnings
warnings.filterwarnings('ignore')


In [329]:
data = pd.read_csv('../data/train.csv')
data = data.dropna()
X = data.drop(['index', 'IC50, mM', 'CC50, mM', 'SI'], axis=1)
y_0 = pd.Series(data['IC50, mM'].astype(float))
y_1 = pd.Series(data['CC50, mM'].astype(float))
y_2 = pd.Series(data['SI'])

In [ ]:
def find_consts(X: pd.DataFrame):
    for x in X:
        std = X[x].std()
        if std == np.float64(0.0):
            yield x

def find_noise(X: pd.DataFrame, y_0, y_1, y_2):
    for x in X.columns.tolist():
        r1 = abs(stats.spearmanr(X[x], y_0)[0])
        r2 = abs(stats.spearmanr(X[x], y_1)[0])
        r3 = abs(stats.spearmanr(X[x], y_2)[0])
        if max(r1, r2, r3) < 0.05:
            yield x

def clean_ejections(data, y: pd.Series):
    data['y'] = y
    lo, hi = list(y.quantile([0.01, 0.99]))
    data = data[(data['y'] > lo) & (data['y'] < hi)]
    return data.drop('y', axis=1), data['y']

def splitting(X, y):
    return train_test_split(X, y, random_state=42)

def fit_predict (Xt, Xp, yt):
    model = catboost.CatBoostRegressor(random_seed=42)
    model.fit(Xt, yt)
    preds = model.predict(Xp)
    return preds

def fit (Xt, yt):
    model = catboost.CatBoostRegressor(random_seed=42)
    model.fit(Xt, yt)
    return model

def fit (model: catboost.CatBoostRegressor, y):
    return model.predict(y)

def get_features(X, y, k=230):
    selector = SelectKBest(score_func=f_regression, k=k)
    selector.fit_transform(X, y)
    indexes = [int(i) for i in selector.get_support(indices=1)]
    kX = X[X.columns[indexes]]
    return kX
def check(y, y_preds):
    return root_mean_squared_error(y, y_preds)


In [331]:
consts, noises = find_consts(X), find_noise(X, y_0, y_1, y_2)
X = X.drop(list(consts) and list(noises), axis=1)

In [332]:
x4y_0 = get_features(X,y_0)
x4y_1 = get_features(X,y_1)
x4y_2 = get_features(X,y_2)

In [333]:
x4y_0, y_0 = clean_ejections(x4y_0, y_0)
x4y_1, y_1 = clean_ejections(x4y_1, y_1)
x4y_2, y_2 = clean_ejections(x4y_2, y_2)

In [334]:
*samples_0, true_0 = splitting(x4y_0, y_0)
*samples_1, true_1 = splitting(x4y_1, y_1)
*samples_2, true_2 = splitting(x4y_2, y_2)

preds_0 = fit_predict(*samples_0)
preds_1 = fit_predict(*samples_1)
preds_2 = fit_predict(*samples_2)

Learning rate set to 0.037242
0:	learn: 297.2118242	total: 8.56ms	remaining: 8.56s
1:	learn: 295.4013755	total: 16.4ms	remaining: 8.17s
2:	learn: 292.8884054	total: 22.4ms	remaining: 7.44s
3:	learn: 290.6348852	total: 28.9ms	remaining: 7.2s
4:	learn: 289.0400793	total: 35.6ms	remaining: 7.09s
5:	learn: 286.6797557	total: 40.9ms	remaining: 6.78s
6:	learn: 285.0655317	total: 47.4ms	remaining: 6.73s
7:	learn: 282.2764212	total: 54.5ms	remaining: 6.76s
8:	learn: 280.3858217	total: 59.4ms	remaining: 6.54s
9:	learn: 278.4406100	total: 66.5ms	remaining: 6.58s
10:	learn: 276.7709739	total: 74.7ms	remaining: 6.71s
11:	learn: 275.5271976	total: 81.8ms	remaining: 6.74s
12:	learn: 274.1289019	total: 88.1ms	remaining: 6.69s
13:	learn: 272.4978185	total: 97.5ms	remaining: 6.87s
14:	learn: 270.3123563	total: 104ms	remaining: 6.8s
15:	learn: 269.2407851	total: 111ms	remaining: 6.83s
16:	learn: 267.5713758	total: 118ms	remaining: 6.83s
17:	learn: 265.6218539	total: 125ms	remaining: 6.84s
18:	learn: 264

In [335]:
results = check(preds_0, true_0), check(preds_1, true_1), check(preds_2, true_2) 

In [338]:
results

(252.92204986221086, 399.7459987774033, 58.45754690903855)

In [ ]:
model_0 = fit(x4y_0, y_0)
model_1 = fit(x4y_1, y_1)
model_2 = fit(x4y_2, y_2)